# 04 — Model Training

Trains and compares classifiers for variant pathogenicity prediction using
`src/model.py`. Outputs the best pipeline to `models/best_pipeline.joblib`
for use in `05_evaluation.ipynb`.

**Covers:**
1. Load features `X`, target `y`, and `groups` from `data/`
2. Train / test split (hold-out last 20 % trios)
3. Train Random Forest baseline
4. Train XGBoost
5. Compare CV metrics across classifiers
6. Select best and save as `models/best_pipeline.joblib`
7. Quick sanity-check on hold-out set

> **Prerequisite:** run `03_feature_engineering.ipynb` first to produce
> `data/features_X.parquet`, `data/features_y.csv`, and `data/features_groups.csv`.

> **MLflow:** runs are tracked under the `variant_pathogenicity` experiment.
> Launch `mlflow ui` in the project root to browse them.

## 1. Imports & Configuration

In [ ]:
import sys, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / "data"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Fraction of trios held out for testing (kept for 05_evaluation.ipynb)
TEST_TRIO_FRACTION = 0.20

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("ROOT:", ROOT)

## 2. Load Features

In [ ]:
X      = pd.read_parquet(DATA_DIR / "features_X.parquet")
y      = pd.read_csv(DATA_DIR / "features_y.csv").squeeze()
groups = pd.read_csv(DATA_DIR / "features_groups.csv").squeeze()

print(f"X : {X.shape}")
print(f"y : {y.shape}  (positive rate = {y.mean():.1%})")
print(f"Unique trios: {groups.nunique()}")

y = y.astype(int)

## 3. Train / Test Split

Hold out the last `TEST_TRIO_FRACTION` of trios (sorted by `trio_id`) as a
chronological test set. This mirrors the strategy used in `05_evaluation.ipynb`.

In [ ]:
all_trios  = sorted(groups.unique())
n_test     = max(1, int(len(all_trios) * TEST_TRIO_FRACTION))
test_trios = all_trios[-n_test:]
train_mask = ~groups.isin(test_trios)
test_mask  =  groups.isin(test_trios)

X_train, y_train, groups_train = X[train_mask], y[train_mask], groups[train_mask]
X_test,  y_test,  groups_test  = X[test_mask],  y[test_mask],  groups[test_mask]

print(f"Train: {X_train.shape[0]:,} rows  ({len(all_trios)-n_test} trios)")
print(f"Test : {X_test.shape[0]:,}  rows  ({n_test} trios: {test_trios})")
print(f"Train positive rate: {y_train.mean():.1%}")
print(f"Test  positive rate: {y_test.mean():.1%}")

## 4. Train Classifiers

Each call to `train()` runs a `RandomizedSearchCV` with `GroupKFold` and logs
results to MLflow. Adjust `n_iter` / `cv_folds` to trade speed for thoroughness.

In [ ]:
from src.model import train, evaluate, DEFAULT_CONFIG

# Shared config overrides — reduce n_iter for quick experimentation;
# increase to 50+ for production runs.
BASE_CFG = {
    "cv_folds":       3,
    "n_iter":         10,
    "search_strategy": "random",
    "scoring":        "f1_macro",
    "calibrate":      True,
    "models_dir":     str(MODELS_DIR),
    "random_state":   42,
    "n_jobs":         -1,
}

pipelines = {}
cv_results = {}

# ── Random Forest ──────────────────────────────────────────────────────────
print("Training Random Forest …")
cfg_rf = {**BASE_CFG, "classifier": "random_forest"}
pipelines["random_forest"] = train(X_train, y_train, groups_train, cfg_rf)
cv_results["random_forest"] = evaluate(pipelines["random_forest"], X_test, y_test)
print(f"  RF   F1-macro={cv_results['random_forest']['f1_macro']:.3f}  "
      f"ROC-AUC={cv_results['random_forest']['roc_auc']:.3f}")

print("")

In [ ]:
# ── XGBoost ────────────────────────────────────────────────────────────────
print("Training XGBoost …")
cfg_xgb = {**BASE_CFG, "classifier": "xgboost"}
try:
    pipelines["xgboost"] = train(X_train, y_train, groups_train, cfg_xgb)
    cv_results["xgboost"] = evaluate(pipelines["xgboost"], X_test, y_test)
    print(f"  XGB  F1-macro={cv_results['xgboost']['f1_macro']:.3f}  "
          f"ROC-AUC={cv_results['xgboost']['roc_auc']:.3f}")
except Exception as e:
    print(f"  XGBoost skipped: {e}")

# ── LightGBM ───────────────────────────────────────────────────────────────
print("Training LightGBM …")
cfg_lgb = {**BASE_CFG, "classifier": "lightgbm"}
try:
    pipelines["lightgbm"] = train(X_train, y_train, groups_train, cfg_lgb)
    cv_results["lightgbm"] = evaluate(pipelines["lightgbm"], X_test, y_test)
    print(f"  LGB  F1-macro={cv_results['lightgbm']['f1_macro']:.3f}  "
          f"ROC-AUC={cv_results['lightgbm']['roc_auc']:.3f}")
except Exception as e:
    print(f"  LightGBM skipped: {e}")

# ── Logistic Regression (baseline) ─────────────────────────────────────────
print("Training Logistic Regression (baseline) …")
cfg_lr = {**BASE_CFG, "classifier": "logistic"}
pipelines["logistic"] = train(X_train, y_train, groups_train, cfg_lr)
cv_results["logistic"] = evaluate(pipelines["logistic"], X_test, y_test)
print(f"  LR   F1-macro={cv_results['logistic']['f1_macro']:.3f}  "
      f"ROC-AUC={cv_results['logistic']['roc_auc']:.3f}")

print("\nAll classifiers trained.")

## 5. Compare CV Metrics

In [ ]:
metric_names = ["accuracy", "f1_macro", "f1_weighted", "roc_auc", "average_precision"]
rows = []
for clf_name, res in cv_results.items():
    row = {"classifier": clf_name}
    row.update({m: res.get(m, float("nan")) for m in metric_names})
    rows.append(row)

compare_df = pd.DataFrame(rows).set_index("classifier")
display(compare_df.round(4).style.highlight_max(axis=0, color="lightgreen"))

# Bar chart
fig, axes = plt.subplots(1, len(metric_names), figsize=(15, 4))
for ax, metric in zip(axes, metric_names):
    compare_df[metric].plot.bar(ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Hold-out test metrics by classifier", y=1.01)
plt.tight_layout()
plt.show()

## 6. Select Best Model & Save `best_pipeline.joblib`

Pick the classifier with the highest `f1_macro` on the hold-out test set and
copy its saved pipeline to `models/best_pipeline.joblib`.

In [ ]:
import joblib

best_clf = compare_df["f1_macro"].idxmax()
print(f"Best classifier: {best_clf}  (F1-macro = {compare_df.loc[best_clf, 'f1_macro']:.4f})")

src_path  = MODELS_DIR / f"{best_clf}_pipeline.joblib"
dest_path = MODELS_DIR / "best_pipeline.joblib"

if src_path.exists():
    shutil.copy(src_path, dest_path)
    print(f"Copied {src_path.name} → {dest_path}")
else:
    # Fallback: save the in-memory pipeline directly
    joblib.dump(pipelines[best_clf], dest_path)
    print(f"Saved pipeline directly → {dest_path}")

# Verify reload
_ = joblib.load(dest_path)
print("Reload verified ✓")

## 7. Hold-Out Sanity Check

Quick classification report and confusion matrix on the hold-out set.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best_pipe = pipelines[best_clf]
y_pred    = best_pipe.predict(X_test)

print(f"Classification report — {best_clf} on hold-out ({len(y_test)} samples):")
print(classification_report(y_test, y_pred, target_names=["Other", "Pathogenic"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["Other", "Pathogenic"],
    cmap="Blues", ax=ax
)
ax.set_title(f"Confusion matrix — {best_clf}")
plt.tight_layout()
plt.show()

print("\nNext steps → run 05_evaluation.ipynb for full SHAP analysis and per-inheritance breakdown.")